# «Закон-граф»: предиктивный анализ норм российского права

Демонстрация адаптированного конвейера (пакет `law_graph/`), переносящего
подход **top-papers-graph** (темпоральный граф причинно-следственных связей
между научными открытиями → прогноз новых связей) на область российского
права:

- **вместо** связей между открытиями — **причинно-следственные связи в виде
  внесения поправок в законы**: «поправка (ФЗ-N от даты) → изменение нормы
  права»;
- **цель** — научить модель **предиктивному анализу норм права**: какие нормы
  вероятно будут изменены.

Запуск — из корня репозитория, офлайн на синтетическом корпусе поправок
(без внешних данных и платных API).

In [ ]:
# 1. Подготовка данных: темпоральный «закон-граф» норм и поправок

from law_graph.corpus_loader import load_synthetic_corpus
from law_graph.amendment_extractor import (
    amendments_from_structured, influence_from_structured,
)
from law_graph.graph_builder import LegalKnowledgeGraph, build_event_stream

syn = load_synthetic_corpus()
kg = LegalKnowledgeGraph()
for e in amendments_from_structured(syn["amendments"]) + influence_from_structured(syn["influence"]):
    kg.add_event(e)
kg.add_norms_from_events()

print(f"норм: {len(kg.norms)}, законов: {len(kg.laws)}, событий: {len(kg.events)}")
print("топ норм по числу поправок:", kg.top_norms_by_amendments(5))


## 2. Извлечение причинно-следственных связей «поправка → изменение нормы»

Из фрагмента правового текста извлекаются события динамики норм (адаптация
`temporal_triplet_extractor` и `temporal_triplet_extractor` из top-papers-graph).

In [ ]:
from law_graph.amendment_extractor import rule_based_amendments_from_text

text = (
    "Федеральный закон от 12.03.2014 № 35-ФЗ.\n"
    "Статья 1229 Гражданского кодекса Российской Федерации изложена в новой редакции.\n"
    "Статья 1270 дополнена пунктом 2.\n"
    "Признать утратившей силу статью 1240."
)
for ev in rule_based_amendments_from_text(text, source_doc="demo.txt", source_law="ГК РФ", default_year=2014):
    print(ev.as_text())


## 3. Хронологическое разбиение (только фактические поправки)

Как в `run_dataset.py`: структурные связи влияния норм (`отсылает_к` и т.п.)
действуют всегда и не участвуют в разбиении по времени — по времени
разбиваются только фактические события изменения норм.

In [ ]:
from law_graph.graph_builder import chronological_split

FACTUAL = {"изменена", "дополнена", "введена", "изложена_в_новой_редакции",
           "признана_утратившей_силу", "исключена", "уточнена"}

events = build_event_stream(kg)
timed = [e for e in events if e.predicate in FACTUAL]
influence = [e for e in events if e.predicate not in FACTUAL]

train, valid, test = chronological_split(timed, train_ratio=0.6, valid_ratio=0.2)
print(f"факт. поправок: {len(timed)} = train {len(train)} / valid {len(valid)} / test {len(test)}")
print(f"структурных связей влияния норм: {len(influence)}")
train_until = max(e.year() for e in train)
print("train_until =", train_until)


## 4. Признаки норм и прогноз «какие нормы вероятно будут изменены»

Обучающий граф = train-поправки + все структурные связи влияния. Метка ставится
по фактическим изменениям в окне `(train_until, train_until+5]` (valid+test).

In [ ]:
from law_graph.graph_builder import build_task_instances
from law_graph.model import fit_model

train_kg = LegalKnowledgeGraph()
for e in train + influence:
    train_kg.add_event(e)
train_kg.add_norms_from_events()

instances = build_task_instances(train_kg, train_until=train_until, predict_window=5)
future_changed = {e.subject for e in valid + test}   # фактические изменения
for inst in instances:
    inst.label = 1 if inst.norm_id in future_changed else 0

model = fit_model(instances, epochs=400)
print("bias =", round(model.bias, 3))
for name, w in model.weights.items():
    print(f"  {name:28s} w={w:+.3f}")

ranked = sorted(instances, key=lambda i: model.predict_proba(i), reverse=True)
print("\nТОП норм по вероятности изменения:")
for i, inst in enumerate(ranked[:5], 1):
    mark = " [ИЗМЕНИТСЯ]" if inst.label == 1 else ""
    print(f"  {i}. {inst.norm_id:32s} p={model.predict_proba(inst):.3f}{mark}")


## 5. Итог

- **Датасет** — `LegalTaskInstance`: признаки динамики нормы до момента T +
  бинарная метка «изменится ли норма в окне (T, T']».
- **Модель** — интерпретируемый скоринг (`LegalChangeHeuristic`) и обучаемая
  логистическая регрессия (`model.fit_model`) с метриками AUC-ROC / precision@k.
- **Результат** — ранжированный список норм, которые вероятно будут изменены.

Полный конвейер из CLI: `python -m law_graph.run_dataset --mode synthetic --out runs/law_demo --export`.
Подробности — `docs/LAW-GRAPH-PIPELINE.md`.